# SDN 制裁名单解析练习（教学版 notebook）

**学什么**：从美国 OFAC 的制裁名单里，把加密货币地址挖出来、建库、统计、查数据质量。
这四步就是链上合规数据管道的最小雏形（对应学习计划的 CP2 前置练习）。

**怎么用这个文件**：
1. 每个 `In [ ]` 框是一个单元格，点进去按 **Shift + Enter** 运行，结果会显示在框下面；
2. **从上往下按顺序跑**——后面的格子会用到前面格子定义的变量；
3. 改代码随便试，跑坏了就重启重来（菜单 Kernel → Restart & Run All）；
4. 哪格看不懂，把这格的代码+讲解一起复制给 DeepSeek，问"逐行解释这段代码在干什么"。

**会遇到两个数据文件**（已在你的 sdn_data 文件夹）：
- `sdn.csv`——平铺格式，19370 行，无表头
- `sdn_advanced.xml`——结构化格式，127MB，生产级管道用的就是它


## 第 1 部分：把没有表头的 CSV 读进来

### 1.1 先裸读，看看它长什么样

`pd.read_csv(..., header=None)` 里的 `header=None` 告诉 pandas：**第一行不是表头，是数据**。
不写这个参数的话，pandas 会自作主张把第一行数据当成列名——这是读无表头 CSV 最常见的坑。


In [2]:
import pandas as pd

CSV_PATH = r'C:\Users\10268\Desktop\sdn_data\sdn.csv'

raw = pd.read_csv(CSV_PATH, header=None, dtype=str, keep_default_na=False)
print('行数 × 列数:', raw.shape)
raw.head(3)

行数 × 列数: (19370, 12)


,0,1,2,3,4,5,6,7,8,9,10,11
0,36,AEROCARIBBEAN AIRLINES,-0-,CUBA,-0-,-0-,-0-,-0-,-0-,-0-,-0-,-0-
1,173,"ANGLO-CARIBBEAN CO., LTD.",-0-,CUBA,-0-,-0-,-0-,-0-,-0-,-0-,-0-,-0-
2,306,BANCO NACIONAL DE CUBA,-0-,CUBA,-0-,-0-,-0-,-0-,-0-,-0-,-0-,a.k.a. 'BNC'.


`dtype=str` 让所有列都按**文本**读进来（防止编号被读成数字、乱七八糟的自动转换）；
`keep_default_na=False` 防止 "NA"、"NULL" 之类字符串被 pandas 擅自当成缺失值。
对这种来源不明的数据，"全部按原样文本读入"是最安全的姿势。

### 1.2 观察一下：12 列分别是什么

pandas 里 `iloc[行号, 列号]` 是按**位置**取数据（`loc` 是按标签）。看看第一行的每一列：


In [2]:
for j in range(12):
    print(f'第 {j} 列: {raw.iloc[0, j]!r}')

第 0 列: '36'
第 1 列: 'AEROCARIBBEAN AIRLINES'
第 2 列: '-0- '
第 3 列: 'CUBA'
第 4 列: '-0- '
第 5 列: '-0- '
第 6 列: '-0- '
第 7 列: '-0- '
第 8 列: '-0- '
第 9 列: '-0- '
第 10 列: '-0- '
第 11 列: '-0- '


OFAC 文档定义了这 12 列的固定含义（顺序 20 多年没变过）。
我们自己补一个表头——**列名照抄官方文档**，这是行业惯例，不要自己发明：


In [3]:
SDN_COLUMNS = [
    'ent_num',        # 实体编号
    'SDN_Name',       # 被制裁主体名称
    'SDN_Type',       # 类型（个人/实体/船只/飞机，'-0-' 表示空）
    'Program',        # 制裁项目（CUBA / CYBER2 / RUSSIA-EO14024 ...）
    'Title',          # 头衔（个人用）
    'Call_Sign',      # 航空器呼号
    'Vess_type',      # 船只类型
    'Tonnage',        # 吨位
    'GRT',            # 总注册吨
    'VTE',            # 吨位单位
    'Other_add_info', # 其他信息
    'Remarks',        # 备注：别名、网站、邮箱、**加密地址**全混在这列
]

df = pd.read_csv(CSV_PATH, header=None, names=SDN_COLUMNS, dtype=str, keep_default_na=False)
print(f'读入 {len(df)} 个被制裁主体')
df.head(2)

读入 19370 个被制裁主体


,ent_num,SDN_Name,SDN_Type,Program,Title,Call_Sign,Vess_type,Tonnage,GRT,VTE,Other_add_info,Remarks
0,36,AEROCARIBBEAN AIRLINES,-0-,CUBA,-0-,-0-,-0-,-0-,-0-,-0-,-0-,-0-
1,173,"ANGLO-CARIBBEAN CO., LTD.",-0-,CUBA,-0-,-0-,-0-,-0-,-0-,-0-,-0-,-0-


## 第 2 部分：筛出含加密地址的记录

### 2.1 先看一条真实的加密地址记录长什么样

第 12 列（`Remarks`）是个大杂烩。我们直接找一条含 `Digital Currency Address` 的记录，
把它的 Remarks 完整打出来——**写解析器之前先看原始数据，永远的第一步**：


In [4]:
mask = df['Remarks'].str.contains('Digital Currency Address', na=False)
print(f'含加密地址的记录：{mask.sum()} 条')

# 取第一条看看
one = df[mask].iloc[0]
print('主体名称:', one['SDN_Name'])
print('--- Remarks 全文 ---')
print(one['Remarks'])

含加密地址的记录：99 条
主体名称: BANK MARKAZI JOMHOURI ISLAMI IRAN
--- Remarks 全文 ---
Additional Sanctions Information - Subject to Secondary Sanctions; Secondary sanctions risk: section 1(b) of Executive Order 13224, as amended by Executive Order 13886; Digital Currency Address - TRX TNiq9AXBp9EjUqhDhrwrfvAA8U3GUQZH81; alt. Digital Currency Address - TRX TTiDLWE6fZK8okMJv6ijg42yrH6W2pjSr9; alt. Digital Currency Address - TRX TAhwhFv3JpK39Nc2m8W5LPCcoTisutiRfp; alt. Digital Currency Address - TRX TJdgB1k6ot3f2nLuZug6D8eD3HavTmzmSK; alt. Digital Currency Address - TRX TXGHxdYbGy574z5hBu4LNzq9NzjZQ9bhUf; alt. Digital Currency Address - TRX TFQbqaNbmq2xsVor2NbufLkYZvxFC9wC7k; Linked To: ISLAMIC REVOLUTIONARY GUARD CORPS (IRGC)-QODS FORCE; Linked To: HIZBALLAH.


注意两个细节：
1. `str.contains(短语, na=False)`：逐行判断"这一格里有没有这个短语"；`na=False` 让空值一律算"没有"，避免报错；
2. 一条记录里挂了**好几个**地址，币种还不一样（`; alt.` 分隔）——所以不能只数"有几行"，得把每个地址都抠出来。

### 2.2 用正则把每个地址的币种抠出来

正则 `Digital Currency Address - ([A-Z0-9]+)` 的读法：
- `Digital Currency Address - ` 是字面匹配（原样找这个短语）
- `([A-Z0-9]+)` 是**捕获组**：匹配"1 个或多个大写字母或数字"并把结果单独存出来
- 所以 `findall` 返回的不是整个匹配，而是括号里那一小段——正好是币种符号

先在一条假数据上练习这个正则，确认理解了再上真实数据：


In [5]:
import re

# 先拿一段编造的数据练手——这个习惯能省掉 90% 的调试时间
practice = ('Website x.com; Digital Currency Address - XBT 1AbC123; '
            'alt. Digital Currency Address - ETH 0xdeadbeef; '
            'Digital Currency Address - USDT TAb12cd')

found = re.findall(r'Digital Currency Address - ([A-Z0-9]+)', practice)
print(found)   # 应该得到 ['XBT', 'ETH', 'USDT']

['XBT', 'ETH', 'USDT']


In [6]:
# 现在对所有 99 条记录做同样的提取，统计每个币种出现多少次
symbol_counts = {}
for remarks in df.loc[mask, 'Remarks']:
    for sym in re.findall(r'Digital Currency Address - ([A-Z0-9]+)', remarks):
        symbol_counts[sym] = symbol_counts.get(sym, 0) + 1

print(f'提取到的加密地址总数：{sum(symbol_counts.values())} 个')
symbol_counts

提取到的加密地址总数：506 个


{'TRX': 80,
 'XMR': 6,
 'USDT': 43,
 'XBT': 262,
 'ETH': 87,
 'LTC': 8,
 'ZEC': 3,
 'DASH': 5,
 'BTG': 1,
 'ETC': 1,
 'XVG': 1,
 'BCH': 2,
 'USDC': 2,
 'SOL': 2,
 'DOGE': 2,
 'BNB': 1}

### 2.3 换 pandas 的写法（了解即可）

字典循环是"初学者写法"，能看懂最重要。同样的事 pandas 有更短的写法——
`.str.count()` 对整列逐格数正则匹配次数，`value_counts()` 自动分组计数。
两种写法结果一样，工作中两种都会见到：


In [ ]:
all_matches = df.loc[mask, 'Remarks'].str.findall(r'Digital Currency Address - ([A-Z0-9]+)')
flat = all_matches.explode()          # 把每行的列表拆成一行一个
flat.value_counts()                   # 按币种计数、自动降序

## 第 3 部分：撞上一个真实的数据质量问题 ⚠️

上面从 CSV 里只挖出了 506 个地址。但官方其实提供了另一个更完整的文件（sdn_advanced.xml），
第三方解析它得到约 1000 个地址。**差了快一倍，钱去哪了？**

### 3.1 线索：统计 Remarks 的长度

如果很多格子的长度"正好"卡在某个整数上，八成是被截断了：


In [7]:
lens = df.loc[mask, 'Remarks'].str.len()
print('含加密地址记录的 Remarks 长度分布：')
print('  恰好 1000 字符:', (lens == 1000).sum(), '条')
print('  不足 1000     :', (lens < 1000).sum(), '条')
print('  最长:', lens.max())

含加密地址记录的 Remarks 长度分布：
  恰好 1000 字符: 26 条
  不足 1000     : 73 条
  最长: 1000


### 3.2 抓现行：看一条被截断的记录

找一条长度恰好 1000 的，看它的**尾部**——地址是 `币种 + 一串字符` 的格式，
如果结尾悬着半个地址，就是实锤：


In [8]:
truncated = df[mask & (lens == 1000)].iloc[0]
print('主体:', truncated['SDN_Name'])
print('--- Remarks 尾部 150 字符 ---')
print(repr(truncated['Remarks'][-150:]))

主体: ISIL KHORASAN
--- Remarks 尾部 150 字符 ---
'P51m9BaHpmCW8H1BJJNQ; alt. Digital Currency Address - TRX TCqKhk7MpPnGH3UfygE1TRzUo92a5tuVy6; alt. Digital Currency Address - TRX TSbGiFzG9xFXUYgGA6Rw'


**实锤**：结尾悬着 `Digital Currency Address - ` 后面什么都没有——地址被拦腰砍断了。

这就是为什么**平铺 CSV 只适合入门，生产管道必须用 sdn_advanced.xml**：
XML 里每个地址是独立的结构化节点，不存在"塞不下被截断"的问题。

顺便说，这个发现本身就是面试弹药："我实测发现 OFAC 平铺 CSV 的 Remarks 在 1000 字符截断，
导致 52% 的加密地址丢失，所以管道选了 advanced XML"——比背十个知识点都管用。


## 第 4 部分：解析 sdn_advanced.xml（生产级数据源）

### 4.1 先理解 XML 的结构（30 秒版）

XML 就是**带标签的层级文本**。OFAC 这份文件的结构长这样（简化）：

```xml
<Sanctions>                          ← 根
  <ReferenceValueSets>               ← 查找表区（在文件前部！）
    <FeatureType ID="345">Digital Currency Address - ETH</FeatureType>
    <FeatureType ID="992">Digital Currency Address - TRX</FeatureType>
    ...
  </ReferenceValueSets>
  <DistinctParty FixedRef="4632">    ← 每个被制裁主体一条
    <Profile>
      <Alias Primary="true">...央行名...</Alias>
      <Feature FeatureTypeID="345">  ← 币种只存 ID：345！
        <FeatureVersion>
          <VersionDetail>0x098B...</VersionDetail>   ← 地址本体
```

两个设计要点：
1. **币种用数字 ID 引用查找表**——所以必须先读完前面的查找表，才知道 345 = ETH；
2. 127MB 的大文件**不能整个读进内存**——用 `iterparse` 流式处理，读完一个主体立刻丢掉一个。


In [9]:
XML_PATH = r'C:\Users\10268\Desktop\sdn_data\sdn_advanced.xml'
NS = '{https://sanctionslistservice.ofac.treas.gov/api/PublicationPreview/exports/ADVANCED_XML}'

# NS 是"命名空间前缀"——这份 XML 所有标签前面都挂了这一串，
# 找标签时必须带上它，不认识它的话先跳过，把它当成固定写法照抄即可
print('命名空间:', NS)

命名空间: {https://sanctionslistservice.ofac.treas.gov/api/PublicationPreview/exports/ADVANCED_XML}


### 4.2 第一步：扫描查找表，建立"币种 ID → 币种符号"对照表

`ET.iterparse(文件, events=('end',))` 的意思：**边读边触发事件**，
每读完一个完整标签就交给我们处理一次——这就是"流式"。
我们在读到 `FeatureTypeValues`（查找表区块）结束的地方 `break`，不再往后读：


In [10]:
import xml.etree.ElementTree as ET

symbol_of = {}
for _, elem in ET.iterparse(XML_PATH, events=('end',)):
    if elem.tag == NS + 'FeatureType' and (elem.text or '').startswith('Digital Currency Address'):
        m = re.search(r'- ([A-Z0-9]+)$', elem.text)     # 取短语最后的币种符号
        if m:
            symbol_of[elem.get('ID')] = m.group(1)
    if elem.tag == NS + 'FeatureTypeValues':
        break    # 查找表到此结束，后面的全是主体记录

print(f'找到 {len(symbol_of)} 个币种 ID')
symbol_of

找到 20 个币种 ID


{'344': 'XBT',
 '345': 'ETH',
 '444': 'XMR',
 '566': 'LTC',
 '686': 'ZEC',
 '687': 'DASH',
 '688': 'BTG',
 '689': 'ETC',
 '706': 'BSV',
 '726': 'BCH',
 '746': 'XVG',
 '887': 'USDT',
 '907': 'XRP',
 '989': 'BNB',
 '992': 'TRX',
 '996': 'DOGE',
 '998': 'USDC',
 '1007': 'ARB',
 '1008': 'BSC',
 '1167': 'SOL'}

### 4.3 第二步：流式处理每个主体（核心循环）

逐行拆开讲（先读代码里的注释，跑完再回来重读一遍，两边对照）：

- `events=('start', 'end')`：多要了 start 事件，只为在文件开头**拿到根节点的引用**；
- 每处理完一个 `DistinctParty`：`elem.clear()` 清空它、`root.clear()` 把它从树上摘掉——
  不做这两步，iterparse 照样会把整棵树留在内存里，127MB 还是会撑爆；
- 主名称藏在 `Primary="true"` 的那个 `Alias` 里；地址在 `Feature` 节点的
  `FeatureVersion/VersionDetail` 里，`find()` 用路径直接定位。


In [11]:
import time
t0 = time.time()

root = None
n_parties = 0
rows = []

for event, elem in ET.iterparse(XML_PATH, events=('start', 'end')):
    if event == 'start':
        if root is None:
            root = elem     # 记住根节点，后面用来清理内存
        continue
    if elem.tag != NS + 'DistinctParty':
        continue

    n_parties += 1

    # 主名称：Primary="true" 的 Alias 下的 NamePartValue 拼起来
    name = ''
    for alias in elem.iter(NS + 'Alias'):
        if alias.get('Primary') == 'true':
            parts = [v.text or '' for v in alias.iter(NS + 'NamePartValue')]
            name = ' '.join(p for p in parts if p)
            break

    # 加密地址：FeatureTypeID 命中对照表的，把地址文本抠出来
    for feature in elem.iter(NS + 'Feature'):
        sym = symbol_of.get(feature.get('FeatureTypeID'))
        if sym is None:
            continue
        detail = feature.find(f'{NS}FeatureVersion/{NS}VersionDetail')
        if detail is not None and detail.text:
            rows.append({
                'address': detail.text.strip(),
                'currency': sym,
                'entity': name,
                'fixed_ref': elem.get('FixedRef'),
            })

    elem.clear()    # 清空这个主体的内容，释放内存
    root.clear()    # 把处理完的节点从树上摘掉

print(f'{n_parties} 个主体，提取 {len(rows)} 个地址，耗时 {time.time()-t0:.1f} 秒')

19369 个主体，提取 1059 个地址，耗时 4.8 秒


### 4.4 统计 + 和 CSV 对账

1059 vs 506——平铺格式丢了多少，一目了然：


In [12]:
addr = pd.DataFrame(rows)
stats_xml = addr['currency'].value_counts()
stats_xml

currency
XBT     534
TRX     254
ETH     120
USDT     94
LTC      14
XMR      11
BCH       7
DASH      5
ZEC       4
SOL       4
USDC      2
DOGE      2
ETC       1
BTG       1
BSV       1
XVG       1
XRP       1
ARB       1
BSC       1
BNB       1
Name: count, dtype: int64

In [13]:
# 对账表：CSV(截断) vs XML(完整)
stats_csv = flat.value_counts()
cmp = pd.DataFrame({'CSV截断': stats_csv, 'XML完整': stats_xml}).fillna(0).astype(int)
cmp['丢失'] = cmp['XML完整'] - cmp['CSV截断']
cmp.sort_values('丢失', ascending=False)

NameError: name 'flat' is not defined

## 第 5 部分：数据质量检查（真实管道的日常）

解析完不查质量就交付 = 裸奔。三个最基本的检查：

### 5.1 重复地址

1059 行里有 16 组重复。**重复不是坏事，是信息**——看看它们是什么：


In [14]:
dups = addr[addr['address'].duplicated(keep=False)].sort_values('address')
print(f'涉及重复的行：{len(dups)} 行')
dups.head(16)

涉及重复的行：29 行


,address,currency,entity,fixed_ref
336,0x19aa5fe80d33a56d56c78e82ea5e50e5d80b4dff,ETH,"SUEX OTC, S.R.O.",33151
344,0x19aa5fe80d33a56d56c78e82ea5e50e5d80b4dff,USDT,"SUEX OTC, S.R.O.",33151
335,0x2f389ce8bd8ff92de3402ffce4691d17fc4f6535,ETH,"SUEX OTC, S.R.O.",33151
343,0x2f389ce8bd8ff92de3402ffce4691d17fc4f6535,USDT,"SUEX OTC, S.R.O.",33151
666,0x4f47bc496083c727c5fbe3ce9cdf2b0f6496270c,BSC,Sim Hyon Sop,42498
665,0x4f47bc496083c727c5fbe3ce9cdf2b0f6496270c,ARB,Sim Hyon Sop,42498
664,0x4f47bc496083c727c5fbe3ce9cdf2b0f6496270c,ETH,Sim Hyon Sop,42498
757,0x983a81ca6FB1e441266D2FbcB7D8E530AC2E05A2,USDC,"Valerian Labs, Inc.",45314
755,0x983a81ca6FB1e441266D2FbcB7D8E530AC2E05A2,ETH,"Valerian Labs, Inc.",45314
756,0x983a81ca6FB1e441266D2FbcB7D8E530AC2E05A2,USDT,"Valerian Labs, Inc.",45314


对着输出，把重复分成三类（每类的含义完全不同）：

| 类型 | 例子 | 含义 |
|---|---|---|
| 同地址、多币种标签 | SUEX 一址同时标 ETH/USDT/XBT | 同一个地址能持有多币种 → **筛查主键应是"地址"，币种只是元数据** |
| 同地址挂在不同主体 | Khatibi 和 Nikaeen 共用一个 XBT 地址 | **地址复用 = 把两个实体关联起来的"链上握手"证据**，调查员的最爱 |
| 纯重复 | Blender.io 同一地址列两次 | 数据源自身的瑕疵，入库时去重即可 |

### 5.2 格式抽查

ETH 地址永远 `0x` 开头、长度 40 的十六进制；TRX 地址永远 `T` 开头。
花 10 秒验一遍，能拦住解析器"错位截取"这种低级错误：


In [15]:
eth = addr.loc[addr['currency'] == 'ETH', 'address']
trx = addr.loc[addr['currency'] == 'TRX', 'address']
print(f'ETH: {len(eth)} 个，全部 0x 开头: {eth.str.startswith("0x").all()}')
print(f'TRX: {len(trx)} 个，全部 T 开头: {trx.str.startswith("T").all()}')

ETH: 120 个，全部 0x 开头: True
TRX: 254 个，全部 T 开头: True


### 5.3 保存"最小制裁地址库"

这就是前面所有步骤的产出——一个合规管道的输入表。
日常业务只需要拿新地址来这张表里查，命中 = 红灯：


In [16]:
OUT = r'C:\Users\10268\Desktop\sdn_data\sdn_addresses.csv'
addr.to_csv(OUT, index=False, encoding='utf-8-sig')
print(f'已保存 {len(addr)} 行 -> sdn_addresses.csv')

已保存 1059 行 -> sdn_addresses.csv


## 第 6 部分：留给你的练习题（动手才算学会）

下面三题从易到难，做出来你就摸到 CP2 管道的门了。**先自己写，卡住了把代码贴给 DeepSeek 问**。

**练习 1（分组统计）**：统计"平均每个主体挂几个地址"，再找出挂地址最多的前 5 个主体。
*提示：`addr.groupby('entity')['address'].count()`，然后 `.sort_values().tail(5)`*

**练习 2（模拟筛查）**：写一个函数 `screen(address)`——传入任意地址，
在 `sdn_addresses.csv` 里精确匹配，命中返回主体名，不命中返回"clean"。
拿库里第 100 行的真地址和一个瞎编的地址各测一次。
*提示：`addr.loc[addr['address'] == address, 'entity']`，注意空结果的分支*

**练习 3（进阶，选做）**：`FixedRef` 相同 = 同一主体。用 `groupby` 找出
"同一主体在多个币种上有地址"的记录，想想这在筛查逻辑上意味着什么。

---

**学完这个 notebook 你掌握了**：无表头 CSV 读入、字符串筛选、正则捕获组、
大文件流式解析、数据质量三查。对应学习计划的位置：CP2 前置练习已完成，
下一个里程碑不变——**CP1（2026.12）：3 个 Dune 看板**。
